# TP5. Synthèse : chiffrer, déchiffrer, casser

Carnet de travail. L'énoncé complet, avec les corrigés, est sur le site :
https://cours.gclab.fr/outils-fondamentaux/tp5-synthese/

Exécutez une cellule avec Maj+Entrée. **Pensez à télécharger ce carnet avant de partir** : menu File, puis Download.


## Exercice 1.1 : les deux conversions

Écrivez `lettre_vers_nombre(c)` et `nombre_vers_lettre(n)`, réciproques l'une de
l'autre sur les majuscules.

**Résultat attendu :** `lettre_vers_nombre("C")` vaut `2`, et
`nombre_vers_lettre(2)` vaut `"C"`.


In [ ]:
def lettre_vers_nombre(c):
    return ord(c) - ord('A')

def nombre_vers_lettre(n):
    return chr(ord('A') + n)

print(lettre_vers_nombre("C"))
print(nombre_vers_lettre(2))


## Exercice 1.2 : nettoyer un texte

Écrivez `nettoyer(texte)` qui met tout en majuscules et ne garde que les lettres de A
à Z, en supprimant espaces, ponctuation et accents.

**Résultat attendu :** `nettoyer("Été, à Paris !")` vaut `"ETEAPARIS"`.


In [ ]:
import unicodedata

def nettoyer(texte):
    result = ""
    for c in unicodedata.normalize('NFD', texte.upper()):
        if 'A' <= c <= 'Z':
            result += c
    return result

print(nettoyer("Eté, à Paris !"))


## Exercice 2.1 : la fonction de chiffrement

Écrivez `chiffrer(texte, a, b)` qui applique $E(x) = (ax + b) \bmod 26$ à chaque
lettre d'un texte déjà nettoyé.

**Résultat attendu :** `chiffrer("ATTAQUEALAUBE", 5, 8)` vaut `"IZZIKECILIENC"`.


In [ ]:
def chiffrer(texte, a, b):
    result = ""
    for c in texte:
        n = lettre_vers_nombre(c)
        result += nombre_vers_lettre((a * n + b) % 26)
    return result

print(chiffrer("ATTAQUEALAUBE", 5, 8))


## Exercice 3.1 : les clés utilisables

Écrivez le code qui donne la liste des valeurs de `a` entre 1 et 25 pour lesquelles le
chiffrement est déchiffrable. Combien y en a-t-il ?

**Résultat attendu :** douze valeurs, toutes impaires et différentes de 13.


In [ ]:
from math import gcd

cles = [a for a in range(1, 26) if gcd(a, 26) == 1]
print(len(cles))
print(cles)


## Exercice 3.2 : l'inverse modulaire

Écrivez `inverse_modulaire(a, n)` qui renvoie l'inverse de `a` modulo `n`, ou lève une
erreur s'il n'existe pas.

**Résultat attendu :** l'inverse de 5 modulo 26 est 21, car $5 \times 21 = 105 =
4 \times 26 + 1$.


In [ ]:
def inverse_modulaire(a, n):
    for i in range(1, n):
        if (a * i) % n == 1:
            return i
    raise ValueError(f"{a} n'a pas d'inverse mod {n}")

print(inverse_modulaire(5, 26))


## Exercice 4.1 : la fonction de déchiffrement

Écrivez `dechiffrer(texte, a, b)`, en refusant les clés invalides. Vérifiez que
déchiffrer ce qu'on a chiffré redonne le message d'origine.

**Résultat attendu :** `dechiffrer(chiffrer(m, 5, 8), 5, 8)` redonne `m` nettoyé.


In [ ]:
from math import gcd

def dechiffrer(texte, a, b):
    if gcd(a, 26) != 1:
        raise ValueError("cle invalide")
    a_inv = inverse_modulaire(a, 26)
    result = ""
    for c in texte:
        n = lettre_vers_nombre(c)
        result += nombre_vers_lettre((a_inv * (n - b)) % 26)
    return result

m = nettoyer("BONJOUR")
print(dechiffrer(chiffrer(m, 5, 8), 5, 8))


## Exercice 5.1 : essayer toutes les clés

Écrivez `toutes_les_clefs(secret)` qui renvoie la liste des 312 déchiffrements
possibles avec leur clé. Retrouvez le message clair à l'œil.

**Résultat attendu :** parmi les 312 propositions, une seule est du français lisible.


In [ ]:
def toutes_les_clefs(secret):
    from math import gcd
    resultats = []
    for a in range(1, 26):
        if gcd(a, 26) == 1:
            for b in range(26):
                resultats.append(((a, b), dechiffrer(secret, a, b)))
    return resultats

secret = chiffrer(nettoyer("BONJOURCOMMENTCAVA"), 5, 8)
candidats = toutes_les_clefs(secret)
print(f"{len(candidats)} propositions")
for (a, b), msg in candidats:
    print(f"a={a} b={b}: {msg}")


## Exercice 6.1 : casser automatiquement

Comptez les occurrences de chaque lettre du message chiffré, puis testez les clés qui
envoient E sur la lettre la plus fréquente. Retrouvez le message sans intervention
humaine.

**Résultat attendu :** le message clair, trouvé parmi une poignée de candidats au
lieu de 312.


In [ ]:
def casser(secret):
    from math import gcd
    freq = {}
    for c in secret:
        freq[c] = freq.get(c, 0) + 1
    lettre_freq = max(freq, key=freq.get)
    n_E = lettre_vers_nombre('E')
    n_freq = lettre_vers_nombre(lettre_freq)
    resultats = []
    for a in range(1, 26):
        if gcd(a, 26) == 1:
            b = (n_freq - a * n_E) % 26
            try:
                resultats.append(((a, b), dechiffrer(secret, a, b)))
            except:
                pass
    return resultats

candidats = casser(secret)
print(f"{len(candidats)} candidats")
for (a, b), msg in candidats:
    print(f"a={a} b={b}: {msg}")


## Exercice 7.1 : l'indice de coïncidence

Écrivez `indice_coincidence(texte)`. Calculez-le sur `texte_long`, sur `secret_long`,
et sur une suite de lettres tirées au hasard de même longueur.

**Résultat attendu :** la même valeur pour le clair et pour le chiffré affine, environ
0,087 sur ce texte court, contre environ 0,039 pour les lettres au hasard.


In [ ]:
import random, string

def indice_coincidence(texte):
    n = len(texte)
    if n < 2:
        return 0
    freq = {}
    for c in texte:
        freq[c] = freq.get(c, 0) + 1
    return sum(f * (f - 1) for f in freq.values()) / (n * (n - 1))

texte_long = nettoyer("le langage python est un langage de programmation tres utilise dans le monde scientifique et dans le developpement logiciel il est connu pour sa syntaxe simple et lisible")
secret_long = chiffrer(texte_long, 5, 8)
aleatoire = ''.join(random.choices(string.ascii_uppercase, k=len(texte_long)))
print(f"clair:          {indice_coincidence(texte_long):.3f}")
print(f"chiffre affine: {indice_coincidence(secret_long):.3f}")
print(f"aleatoire:      {indice_coincidence(aleatoire):.3f}")


## Exercice 8.1 : chiffrer par Vigenère

Écrivez `vigenere(texte, cle)`. Chiffrez `texte_long` avec la clé `"CLE"`, et calculez
l'indice de coïncidence du résultat.

**Résultat attendu :** un indice d'environ 0,055, intermédiaire entre le français et
le hasard. L'attaque par fréquences de l'étape 6 ne s'applique plus telle quelle.


In [ ]:
def vigenere(texte, cle):
    cle = nettoyer(cle)
    result = ""
    for i, c in enumerate(texte):
        k = lettre_vers_nombre(cle[i % len(cle)])
        result += nombre_vers_lettre((lettre_vers_nombre(c) + k) % 26)
    return result

chiffre_vig = vigenere(texte_long, "CLE")
print(f"IC vigenere: {indice_coincidence(chiffre_vig):.3f}")


## Exercice 8.2 : retrouver la longueur de la clé

Si la clé a $L$ lettres, alors les lettres du chiffré de rangs $0, L, 2L, \dots$
ont toutes subi le même décalage : c'est un chiffre de César, et son indice de
coïncidence est celui du français. Pour chaque longueur candidate `L` de 1 à 7,
découpez le chiffré en `L` colonnes, calculez l'indice moyen des colonnes, et repérez
les longueurs où il remonte.

**Résultat attendu :** l'indice moyen bondit à environ 0,09 pour `L = 3` et
`L = 6`, et reste vers 0,055 pour les autres longueurs. La clé a donc trois lettres,
ou un multiple de trois.


In [ ]:
for L in range(1, 8):
    colonnes = [chiffre_vig[i::L] for i in range(L)]
    ic_moyen = sum(indice_coincidence(col) for col in colonnes) / L
    print(f"L={L}: IC moyen={ic_moyen:.3f}")
